# BSRBF-KAN + KAN-Mixer với tăng cường RSS theo thiết bị: UJIIndoorLoc

Notebook v5 xuất phát từ `BSRBF_KAN.ipynb` của bạn. Huấn luyện BSRBF-KAN **riêng từng tòa nhà** đúng kiến trúc `[520,512,512,256,256,128,128,64,64,2]`, rồi đóng băng mạng ấy và huấn luyện KAN-Mixer để **chỉnh tọa độ**, thay vì thay toàn bộ mạng bằng Mixer mới. Khối Mixer nhận 64 đặc trưng áp chót và RSS gốc. Đầu hiệu chỉnh khởi tạo 0: trước khi học, dự đoán ghép bằng đúng dự đoán BSRBF-KAN của *lần chạy này*.

**Đánh giá:** `trainingData.csv` chia theo **vị trí đo (tòa, tầng, cặp tọa độ)**: các mẫu tại cùng vị trí không thể xuất hiện ở cả train và validation. `validationData.csv` chính thức được đánh giá một lần cho mỗi mô hình, sau khi chọn checkpoint và hệ số hiệu chỉnh bằng validation nội bộ. Sai số là khoảng cách Euclid trung bình, đơn vị mét. Các mô hình theo từng tòa cần biết tòa nhà khi suy luận; notebook dùng nhãn tòa có sẵn trong UJI để so sánh công bằng với baseline gốc. Muốn triển khai thực tế phải thêm bộ phân loại tòa riêng và đo cả lỗi của nó.

Mốc **<5 m là mục tiêu**, không phải kết quả của notebook này. Chạy toàn bộ trên Google Colab GPU và đọc bảng test cuối trước khi kết luận. Notebook không chọn mô hình hoặc hệ số dựa trên official test.

**Thử nghiệm mới:** biến đổi RSS trong train bằng lệch gain chung, nhiễu từng AP và mất AP ngẫu nhiên; validation/test vẫn dùng RSS gốc. Mục đích là tăng khả năng chịu khác biệt thiết bị và thời gian đo. Đây là giả thuyết phải kiểm tra, không phải kết quả đã đạt <5 m. So sánh v5 với v4 trên cùng official test, không chọn siêu tham số theo test.


In [ ]:
import os
import math
import copy
import random
import shutil
import zipfile
import urllib.request
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
from pathlib import Path
import json


In [ ]:
# =============================================================================
# CONFIG
# =============================================================================

SEED = 42

BATCH_SIZE = 64

GRID_SIZE = 4
SPLINE_ORDER = 3

# Stage 1: train last layers first
STAGE1_EPOCHS = 30
STAGE1_LR = 3e-4

# Stage 2: full-network fine tuning
STAGE2_EPOCHS = 80
PATIENCE = 15

VAL_SIZE = 0.15
MIXER_EPOCHS = 100
MIXER_PATIENCE = 18
MIXER_LR = 3e-4
MIXER_BLOCKS = 2
MIXER_BATCH_SIZE = 128
SAVE_DIR = Path('/content/bsrbf_device_robust_v5')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

HIDDEN = [
    512, 512,
    256, 256,
    128, 128,
    64, 64
]

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("=" * 90)
print("BSRBF-KAN × OFFICIAL UJIIndoorLoc")
print("TWO-STAGE TRAINING")
print("=" * 90)

print("PyTorch :", torch.__version__)
print("Device  :", DEVICE)

if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))

print("=" * 90)

In [ ]:
class RadialBasisFunction(nn.Module):

    def __init__(
        self,
        grid_min=-1.5,
        grid_max=1.5,
        num_grids=8,
        denominator=None
    ):
        super().__init__()

        grid = torch.linspace(
            grid_min,
            grid_max,
            num_grids
        )

        self.grid = nn.Parameter(
            grid,
            requires_grad=False
        )

        self.denominator = (
            denominator
            or
            (grid_max - grid_min) / (num_grids - 1)
        )

    def forward(self, x):

        return torch.exp(
            -(
                (
                    x[..., None]
                    - self.grid
                )
                / self.denominator
            ) ** 2
        )

In [ ]:
class BSRBF_KANLayer(nn.Module):

    def __init__(
        self,
        input_dim,
        output_dim,
        grid_size=5,
        spline_order=3,
        base_activation=nn.SiLU,
        grid_range=(-1.5, 1.5)
    ):
        super().__init__()

        self.input_dim = input_dim
        self.output_dim = output_dim

        self.grid_size = grid_size
        self.spline_order = spline_order

        self.layernorm = nn.LayerNorm(
            input_dim
        )

        self.base_activation = (
            base_activation()
        )

        # ================================================================
        # Base branch
        # ================================================================

        self.base_weight = nn.Parameter(
            torch.empty(
                output_dim,
                input_dim
            )
        )

        nn.init.kaiming_uniform_(
            self.base_weight,
            a=math.sqrt(5)
        )

        # ================================================================
        # BSRBF branch
        # ================================================================

        self.spline_weight = nn.Parameter(
            torch.empty(
                output_dim,
                input_dim * (
                    grid_size
                    + spline_order
                )
            )
        )

        nn.init.kaiming_uniform_(
            self.spline_weight,
            a=math.sqrt(5)
        )

        self.rbf = RadialBasisFunction(
            grid_min=grid_range[0],
            grid_max=grid_range[1],
            num_grids=(
                grid_size
                + spline_order
            )
        )

        h = (
            grid_range[1]
            - grid_range[0]
        ) / grid_size

        grid = (
            (
                torch.arange(
                    -spline_order,
                    grid_size
                    + spline_order
                    + 1
                )
                * h
                + grid_range[0]
            )
            .expand(
                input_dim,
                -1
            )
            .contiguous()
        )

        self.register_buffer(
            "grid",
            grid
        )

    # =====================================================================
    # B-SPLINE
    # =====================================================================

    def b_splines(self, x):

        assert x.dim() == 2
        assert x.size(1) == self.input_dim

        grid = self.grid

        x = x.unsqueeze(-1)

        bases = (
            (
                x >= grid[:, :-1]
            )
            &
            (
                x < grid[:, 1:]
            )
        ).to(x.dtype)

        for k in range(
            1,
            self.spline_order + 1
        ):

            bases = (
                (
                    (
                        x
                        - grid[:, :-(k + 1)]
                    )
                    /
                    (
                        grid[:, k:-1]
                        - grid[:, :-(k + 1)]
                    )
                )
                * bases[:, :, :-1]
            ) + (
                (
                    (
                        grid[:, k + 1:]
                        - x
                    )
                    /
                    (
                        grid[:, k + 1:]
                        - grid[:, 1:-k]
                    )
                )
                * bases[:, :, 1:]
            )

        return bases.contiguous()

    # =====================================================================
    # FORWARD
    # =====================================================================

    def forward(self, x):

        x = self.layernorm(x)

        # Base branch
        base_output = F.linear(
            self.base_activation(x),
            self.base_weight
        )

        # B-spline branch
        bs_output = (
            self
            .b_splines(x)
            .reshape(
                x.size(0),
                -1
            )
        )

        # RBF branch
        rbf_output = (
            self
            .rbf(x)
            .reshape(
                x.size(0),
                -1
            )
        )

        # Combined BSRBF
        bsrbf_output = (
            bs_output
            + rbf_output
        )

        bsrbf_output = F.linear(
            bsrbf_output,
            self.spline_weight
        )

        return (
            base_output
            + bsrbf_output
        )

In [ ]:
class BSRBF_KAN(nn.Module):

    def __init__(
        self,
        layers_hidden,
        grid_size=5,
        spline_order=3,
        base_activation=nn.SiLU
    ):
        super().__init__()

        self.layers = nn.ModuleList()

        for input_dim, output_dim in zip(
            layers_hidden,
            layers_hidden[1:]
        ):

            self.layers.append(
                BSRBF_KANLayer(
                    input_dim,
                    output_dim,
                    grid_size=grid_size,
                    spline_order=spline_order,
                    base_activation=base_activation
                )
            )

    def forward(self, x):

        for layer in self.layers:
            x = layer(x)

        return x


print("BSRBF-KAN model loaded.")

In [ ]:
BASE_DIR = "/content/UJIndoorLoc"  # đặt CSV vào đây nếu đã tải trước

TRAIN_PATH = os.path.join(
    BASE_DIR,
    "trainingData.csv"
)

TEST_PATH = os.path.join(
    BASE_DIR,
    "validationData.csv"
)

if not (
    os.path.exists(TRAIN_PATH)
    and
    os.path.exists(TEST_PATH)
):

    print("\nDownloading UJIIndoorLoc...")

    zip_path = "/content/uji.zip"

    url = (
        "https://archive.ics.uci.edu/"
        "static/public/310/"
        "ujiindoorloc.zip"
    )

    urllib.request.urlretrieve(
        url,
        zip_path
    )

    temp_dir = "/content/UJI_TEMP"

    if os.path.exists(temp_dir):
        shutil.rmtree(temp_dir)

    with zipfile.ZipFile(
        zip_path,
        "r"
    ) as z:

        z.extractall(
            temp_dir
        )

    found_train = None
    found_test = None

    for root, dirs, files in os.walk(
        temp_dir
    ):

        for f in files:

            if f.lower() == "trainingdata.csv":

                found_train = os.path.join(
                    root,
                    f
                )

            elif f.lower() == "validationdata.csv":

                found_test = os.path.join(
                    root,
                    f
                )

    if found_train is None or found_test is None:

        raise FileNotFoundError(
            "UJIIndoorLoc CSV files not found."
        )

    os.makedirs(
        BASE_DIR,
        exist_ok=True
    )

    shutil.copy(
        found_train,
        TRAIN_PATH
    )

    shutil.copy(
        found_test,
        TEST_PATH
    )

print("\nTraining file :", TRAIN_PATH)
print("Test file     :", TEST_PATH)

In [ ]:
train_df = pd.read_csv(
    TRAIN_PATH
)

test_df = pd.read_csv(
    TEST_PATH
)

WAP_COLS = [
    c
    for c in train_df.columns
    if c.upper().startswith("WAP")
]

print("\n" + "=" * 90)
print("DATASET")
print("=" * 90)

print(
    "Training samples:",
    len(train_df)
)

print(
    "Official test  :",
    len(test_df)
)

print(
    "WAPs           :",
    len(WAP_COLS)
)

print(
    "Buildings      :",
    sorted(
        train_df.BUILDINGID.unique()
    )
)

print("=" * 90)

In [ ]:
def preprocess_rss(df):

    X = (
        df[WAP_COLS]
        .to_numpy(
            dtype=np.float32
        )
    )

    # Missing AP
    X[X == 100] = -110.0

    # Keep realistic RSS interval
    X = np.clip(
        X,
        -110.0,
        -30.0
    )

    # [-110,-30] -> [-1,+1]
    X = (
        2.0
        *
        (
            (X + 110.0)
            /
            80.0
        )
        -
        1.0
    )

    return X.astype(
        np.float32
    )


print("RSS preprocessing ready.")

In [ ]:
class LocalizationLoss(nn.Module):

    def __init__(
        self,
        alpha=0.25
    ):
        super().__init__()

        self.alpha = alpha

    def forward(
        self,
        pred,
        target
    ):

        dist = torch.sqrt(
            torch.sum(
                (
                    pred
                    - target
                ) ** 2,
                dim=1
            )
            + 1e-8
        )

        euclidean_loss = (
            dist.mean()
        )

        mse_loss = F.mse_loss(
            pred,
            target
        )

        return (
            euclidean_loss
            +
            self.alpha
            * mse_loss
        )


print("LocalizationLoss ready.")

In [ ]:
# =============================================================================
# METRICS
# =============================================================================

def calculate_metrics(
    true_xy,
    pred_xy
):

    true_xy = np.asarray(
        true_xy,
        dtype=np.float64
    )

    pred_xy = np.asarray(
        pred_xy,
        dtype=np.float64
    )

    # Euclidean positioning error
    errors = np.linalg.norm(
        true_xy - pred_xy,
        axis=1
    )

    return {
        "errors": errors,

        "mean":
            float(np.mean(errors)),

        "median":
            float(np.median(errors)),

        "p75":
            float(np.percentile(errors, 75)),

        "p90":
            float(np.percentile(errors, 90)),

        "p95":
            float(np.percentile(errors, 95)),

        "cdf30":
            float(
                np.mean(errors <= 30)
                * 100
            )
    }


print("calculate_metrics() loaded successfully.")

In [ ]:
# Tăng cường RSS chỉ cho TRAIN: mô phỏng lệch gain điện thoại và AP bị bỏ lỡ.
# RSS đã được đưa về [-1, 1]; 0.025 tương ứng khoảng 1 dBm.
RSS_GAIN_STD = 0.10       # khoảng 4 dBm
RSS_AP_NOISE_STD = 0.05   # khoảng 2 dBm
RSS_DROPOUT_PROB = 0.06

class DeviceAugmentedRSSDataset(torch.utils.data.Dataset):
    def __init__(self, x, y):
        self.x = torch.as_tensor(x, dtype=torch.float32)
        self.y = torch.as_tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        rss = self.x[idx].clone()
        observed = rss > -0.999
        gain = torch.randn(()) * RSS_GAIN_STD
        ap_noise = torch.randn_like(rss) * RSS_AP_NOISE_STD
        rss = torch.where(observed, (rss + gain + ap_noise).clamp(-1, 1), rss)
        dropped = (torch.rand_like(rss) < RSS_DROPOUT_PROB) & observed
        rss[dropped] = -1.0
        return rss, self.y[idx]

print('RSS train augmentation ready; validation/test unchanged.')


In [ ]:
def train_building(building_id):

    print("\n\n" + "#" * 90)
    print(f"BUILDING {building_id}")
    print("#" * 90)

    # =====================================================================
    # 1. BUILDING DATA
    # =====================================================================

    building_train = (
        train_df[
            train_df.BUILDINGID == building_id
        ]
        .reset_index(drop=True)
    )

    building_test = (
        test_df[
            test_df.BUILDINGID == building_id
        ]
        .reset_index(drop=True)
    )

    # =====================================================================
    # 2. INTERNAL TRAIN / VALIDATION SPLIT
    # =====================================================================

    indices = np.arange(
        len(building_train)
    )

    rp = (
        building_train['FLOOR'].astype(str) + '_' +
        building_train['LONGITUDE'].round(3).astype(str) + '_' +
        building_train['LATITUDE'].round(3).astype(str)
    ).to_numpy()
    train_idx, val_idx = next(GroupShuffleSplit(
        n_splits=1, test_size=VAL_SIZE, random_state=SEED
    ).split(indices, groups=rp))
    assert set(rp[train_idx]).isdisjoint(set(rp[val_idx]))

    train_part = (
        building_train
        .iloc[train_idx]
        .reset_index(drop=True)
    )

    val_part = (
        building_train
        .iloc[val_idx]
        .reset_index(drop=True)
    )

    # =====================================================================
    # 3. INPUT
    # =====================================================================

    X_train = preprocess_rss(
        train_part
    )

    X_val = preprocess_rss(
        val_part
    )

    X_test = preprocess_rss(
        building_test
    )

    # =====================================================================
    # 4. TARGET
    # =====================================================================

    target_cols = [
        "LONGITUDE",
        "LATITUDE"
    ]

    y_train = (
        train_part[target_cols]
        .to_numpy(
            dtype=np.float64
        )
    )

    y_val = (
        val_part[target_cols]
        .to_numpy(
            dtype=np.float64
        )
    )

    y_test = (
        building_test[target_cols]
        .to_numpy(
            dtype=np.float64
        )
    )

    # =====================================================================
    # 5. TARGET NORMALIZATION
    # =====================================================================

    scaler = StandardScaler()

    y_train_norm = (
        scaler
        .fit_transform(
            y_train
        )
        .astype(
            np.float32
        )
    )

    y_val_norm = (
        scaler
        .transform(
            y_val
        )
        .astype(
            np.float32
        )
    )

    y_test_norm = (
        scaler
        .transform(
            y_test
        )
        .astype(
            np.float32
        )
    )

    print(
        "Train       :",
        len(X_train)
    )

    print(
        "Internal val:",
        len(X_val)
    )

    print(
        "Official test:",
        len(X_test)
    )

    # =====================================================================
    # 6. LOADERS
    # =====================================================================

    train_loader = DataLoader(
        DeviceAugmentedRSSDataset(X_train, y_train_norm),
        batch_size=BATCH_SIZE,
        shuffle=True,
        drop_last=False
    )

    val_loader = DataLoader(
        TensorDataset(
            torch.tensor(X_val),
            torch.tensor(y_val_norm)
        ),
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    test_loader = DataLoader(
        TensorDataset(
            torch.tensor(X_test),
            torch.tensor(y_test_norm)
        ),
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    # =====================================================================
    # 7. MODEL
    # =====================================================================

    architecture = (
        [len(WAP_COLS)]
        + HIDDEN
        + [2]
    )

    print(
        "Architecture:",
        architecture
    )

    model = BSRBF_KAN(
        architecture,
        grid_size=GRID_SIZE,
        spline_order=SPLINE_ORDER
    ).to(DEVICE)

    print(
        "Parameters:",
        f"{sum(p.numel() for p in model.parameters()):,}"
    )

    criterion = LocalizationLoss(
        alpha=0.25
    )

    # =====================================================================
    # 8. EVALUATION FUNCTION
    # =====================================================================

    @torch.no_grad()
    def evaluate(loader, true_xy):

        model.eval()

        outputs = []

        for xb, _ in loader:

            xb = xb.to(DEVICE)

            pred = model(xb)

            outputs.append(
                pred
                .cpu()
                .numpy()
            )

        pred_norm = np.concatenate(
            outputs,
            axis=0
        )

        pred_xy = scaler.inverse_transform(
            pred_norm.astype(
                np.float64
            )
        )

        metrics = calculate_metrics(
            true_xy,
            pred_xy
        )

        pred_std = np.std(
            pred_norm.astype(
                np.float64
            ),
            axis=0
        )

        return (
            metrics,
            pred_xy,
            pred_norm,
            pred_std
        )

    # =====================================================================
    # 9. STAGE 1
    # Train last 3 BSRBF layers.
    # Earlier layers are frozen.
    # =====================================================================

    print("\n" + "=" * 70)
    print("STAGE 1 — TRAIN LAST 3 BSRBF LAYERS")
    print("=" * 70)

    for p in model.parameters():
        p.requires_grad = False

    for layer in model.layers[-3:]:

        for p in layer.parameters():
            p.requires_grad = True

    optimizer = torch.optim.Adam(
        filter(
            lambda p: p.requires_grad,
            model.parameters()
        ),
        lr=STAGE1_LR
    )

    best_stage1 = float("inf")
    best_stage1_state = None

    for epoch in range(
        1,
        STAGE1_EPOCHS + 1
    ):

        model.train()

        running_loss = 0.0
        n = 0

        for xb, yb in train_loader:

            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad(
                set_to_none=True
            )

            pred = model(xb)

            loss = criterion(
                pred,
                yb
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                filter(
                    lambda p: p.requires_grad,
                    model.parameters()
                ),
                5.0
            )

            optimizer.step()

            running_loss += (
                loss.item()
                * xb.size(0)
            )

            n += xb.size(0)

        val_metrics, _, _, pred_std = evaluate(
            val_loader,
            y_val
        )

        val_error = (
            val_metrics["mean"]
        )

        if val_error < best_stage1:

            best_stage1 = val_error

            best_stage1_state = copy.deepcopy(
                model.state_dict()
            )

        if epoch == 1 or epoch % 5 == 0:

            print(
                f"S1 | "
                f"Epoch {epoch:03d}/{STAGE1_EPOCHS} | "
                f"Loss {running_loss/n:.5f} | "
                f"Val {val_error:.3f} m | "
                f"PredSTD "
                f"({pred_std[0]:.4f}, "
                f"{pred_std[1]:.4f})"
            )

    model.load_state_dict(
        best_stage1_state
    )

    print(
        "\nStage-1 best validation:",
        f"{best_stage1:.3f} m"
    )

    # =====================================================================
    # 10. STAGE 2
    # Full-network fine-tuning
    #
    # early layers : 1e-5
    # middle       : 3e-5
    # final        : 1e-4
    # =====================================================================

    print("\n" + "=" * 70)
    print("STAGE 2 — FULL NETWORK FINE-TUNING")
    print("=" * 70)

    for p in model.parameters():
        p.requires_grad = True

    groups = []

    total_layers = len(
        model.layers
    )

    for i, layer in enumerate(
        model.layers
    ):

        if i < 3:
            lr = 1e-5

        elif i < 6:
            lr = 3e-5

        else:
            lr = 1e-4

        groups.append({
            "params":
                layer.parameters(),

            "lr":
                lr
        })

    optimizer = torch.optim.AdamW(
        groups,
        weight_decay=1e-5
    )

    scheduler = (
        torch.optim.lr_scheduler
        .ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=0.5,
            patience=5,
            min_lr=1e-7
        )
    )

    best_val = float("inf")
    best_state = None
    best_epoch = 0
    wait = 0

    history = {
        "loss": [],
        "val_error": [],
        "std_x": [],
        "std_y": []
    }

    for epoch in range(
        1,
        STAGE2_EPOCHS + 1
    ):

        model.train()

        running_loss = 0.0
        n = 0

        for xb, yb in train_loader:

            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad(
                set_to_none=True
            )

            pred = model(xb)

            loss = criterion(
                pred,
                yb
            )

            if not torch.isfinite(loss):

                raise RuntimeError(
                    "Non-finite loss."
                )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=5.0
            )

            optimizer.step()

            running_loss += (
                loss.item()
                * xb.size(0)
            )

            n += xb.size(0)

        train_loss = (
            running_loss
            / n
        )

        val_metrics, _, _, pred_std = evaluate(
            val_loader,
            y_val
        )

        val_error = (
            val_metrics["mean"]
        )

        scheduler.step(
            val_error
        )

        history["loss"].append(
            train_loss
        )

        history["val_error"].append(
            val_error
        )

        history["std_x"].append(
            pred_std[0]
        )

        history["std_y"].append(
            pred_std[1]
        )

        if val_error < best_val:

            best_val = val_error
            best_epoch = epoch

            best_state = copy.deepcopy(
                model.state_dict()
            )

            wait = 0

        else:
            wait += 1

        if epoch == 1 or epoch % 5 == 0:

            last_lr = (
                optimizer
                .param_groups[-1]["lr"]
            )

            print(
                f"S2 | "
                f"Epoch {epoch:03d}/{STAGE2_EPOCHS} | "
                f"Loss {train_loss:.5f} | "
                f"Val {val_error:.3f} m | "
                f"PredSTD "
                f"({pred_std[0]:.4f}, "
                f"{pred_std[1]:.4f}) | "
                f"LR {last_lr:.2e}"
            )

        if wait >= PATIENCE:

            print(
                f"\nEarly stopping "
                f"at epoch {epoch}"
            )

            break

    # =====================================================================
    # 11. RESTORE BEST INTERNAL VALIDATION CHECKPOINT
    # =====================================================================

    model.load_state_dict(
        best_state
    )

    # =====================================================================
    # 12. OFFICIAL TEST
    # =====================================================================

    test_metrics, pred_xy, pred_norm, pred_std = evaluate(
        test_loader,
        y_test
    )

    true_norm_std = np.std(
        y_test_norm.astype(
            np.float64
        ),
        axis=0
    )

    # =====================================================================
    # 13. DIAGNOSTIC
    # =====================================================================

    print("\n" + "-" * 70)
    print("FINAL PREDICTION DIAGNOSTIC")
    print("-" * 70)

    print(
        "TRUE normalized STD:",
        true_norm_std
    )

    print(
        "PRED normalized STD:",
        pred_std
    )

    print("\nFirst 5 predictions:")

    for i in range(
        min(5, len(y_test))
    ):

        error = np.linalg.norm(
            y_test[i]
            - pred_xy[i]
        )

        print(
            f"{i:03d} | "
            f"TRUE "
            f"{np.round(y_test[i],2)} | "
            f"PRED "
            f"{np.round(pred_xy[i],2)} | "
            f"ERR "
            f"{error:.2f} m"
        )

    # =====================================================================
    # 14. FINAL RESULT
    # =====================================================================

    print("\n" + "=" * 70)

    print(
        f"BUILDING {building_id} FINAL TEST"
    )

    print("=" * 70)

    print(
        f"Best internal epoch : "
        f"{best_epoch}"
    )

    print(
        f"Internal val error  : "
        f"{best_val:.3f} m"
    )

    print(
        f"Test mean error     : "
        f"{test_metrics['mean']:.3f} m"
    )

    print(
        f"Median              : "
        f"{test_metrics['median']:.3f} m"
    )

    print(
        f"P75                 : "
        f"{test_metrics['p75']:.3f} m"
    )

    print(
        f"P90                 : "
        f"{test_metrics['p90']:.3f} m"
    )

    print(
        f"P95                 : "
        f"{test_metrics['p95']:.3f} m"
    )

    print(
        f"CDF <= 30m          : "
        f"{test_metrics['cdf30']:.2f}%"
    )

    print(
        f"Prediction STD      : "
        f"{pred_std}"
    )

    print("=" * 70)

    return {
        "building":
            building_id,

        "train":
            len(X_train),

        "val":
            len(X_val),

        "test":
            len(X_test),

        "mean":
            test_metrics["mean"],

        "median":
            test_metrics["median"],

        "p75":
            test_metrics["p75"],

        "p90":
            test_metrics["p90"],

        "p95":
            test_metrics["p95"],

        "cdf30":
            test_metrics["cdf30"],

        "errors":
            test_metrics["errors"],

        "pred_std_x":
            pred_std[0],

        "pred_std_y":
            pred_std[1],

        "best_epoch":
            best_epoch,

        "history": history,
        "model": model,
        "scaler": scaler,
        "train_idx": train_idx,
        "val_idx": val_idx,
        "building_train": building_train,
        "building_test": building_test,
        "baseline_test_pred": pred_xy,
        "baseline_val_m": best_val,
    }


print("train_building() ready.")

In [ ]:
# Huấn luyện baseline gốc riêng từng tòa, với validation tách theo vị trí.
baseline_results = []
for building_id in sorted(train_df.BUILDINGID.unique()):
    result = train_building(int(building_id))
    baseline_results.append(result)
    if torch.cuda.is_available(): torch.cuda.empty_cache()

baseline_errors = np.concatenate([r['errors'] for r in baseline_results])
print('BSRBF-KAN | official test mean Euclidean:', float(baseline_errors.mean()), 'm')


## KAN-Mixer hiệu chỉnh

Trọng số baseline được đóng băng. Alpha và epoch được chọn chỉ từ validation tách theo vị trí. Alpha 0 bảo đảm lựa chọn baseline nếu Mixer không cải thiện trên validation. Điều này **không bảo đảm** test sẽ cải thiện.

In [ ]:
# Residual KAN-Mixer trên đặc trưng của BSRBF-KAN đã học.
class SmallKANMixerBlock(nn.Module):
    def __init__(self, tokens=16, channels=8):
        super().__init__()
        self.token_norm = nn.LayerNorm(channels)
        self.token_net = nn.Sequential(
            BSRBF_KANLayer(tokens, 32, grid_size=GRID_SIZE, spline_order=SPLINE_ORDER),
            BSRBF_KANLayer(32, tokens, grid_size=GRID_SIZE, spline_order=SPLINE_ORDER))
        self.channel_norm = nn.LayerNorm(channels)
        self.channel_net = nn.Sequential(
            BSRBF_KANLayer(channels, 16, grid_size=GRID_SIZE, spline_order=SPLINE_ORDER),
            BSRBF_KANLayer(16, channels, grid_size=GRID_SIZE, spline_order=SPLINE_ORDER))

    def forward(self, x):
        batch, tokens, channels = x.shape
        t = self.token_norm(x).transpose(1, 2).reshape(-1, tokens)
        x = x + self.token_net(t).reshape(batch, channels, tokens).transpose(1, 2)
        c = self.channel_norm(x).reshape(-1, channels)
        return x + self.channel_net(c).reshape(batch, tokens, channels)


class ResidualKANMixer(nn.Module):
    def __init__(self, n_blocks=MIXER_BLOCKS):
        super().__init__()
        self.rss_projection = nn.Sequential(nn.Linear(520, 64), nn.LayerNorm(64))
        self.blocks = nn.Sequential(*[SmallKANMixerBlock() for _ in range(n_blocks)])
        self.head = nn.Sequential(nn.LayerNorm(128), nn.Linear(128, 64),
                                  nn.SiLU(), nn.Linear(64, 2))
        nn.init.zeros_(self.head[-1].weight)
        nn.init.zeros_(self.head[-1].bias)

    def forward(self, rssi, features):
        tokens = torch.cat([features, self.rss_projection(rssi)], dim=1).reshape(-1, 16, 8)
        return self.head(self.blocks(tokens).flatten(1))


@torch.no_grad()
def baseline_features(model, x, batch_size=128):
    model.eval()
    hidden, pred = [], []
    for start in range(0, len(x), batch_size):
        xb = torch.as_tensor(x[start:start+batch_size], device=DEVICE)
        for layer in model.layers[:-1]: xb = layer(xb)
        hidden.append(xb.cpu())
        pred.append(model.layers[-1](xb).cpu())
    return torch.cat(hidden), torch.cat(pred)


def mixer_predictions(mixer, loader, alpha=1.0):
    mixer.eval()
    predictions = []
    with torch.no_grad():
        for rss, feat, base, _ in loader:
            rss, feat = rss.to(DEVICE), feat.to(DEVICE)
            predictions.append((base.to(DEVICE) + alpha*mixer(rss, feat)).cpu().numpy())
    return np.concatenate(predictions)


def mean_meters(pred_norm, truth_world, scaler):
    pred_world = scaler.inverse_transform(pred_norm.astype(np.float64))
    distances = np.linalg.norm(pred_world - truth_world, axis=1)
    return float(distances.mean()), distances


def train_residual_mixer(result):
    bid = result['building']
    frame, test_frame = result['building_train'], result['building_test']
    ti, vi, scaler = result['train_idx'], result['val_idx'], result['scaler']
    x = preprocess_rss(frame)
    xt = preprocess_rss(test_frame)
    y = frame[['LONGITUDE','LATITUDE']].to_numpy(np.float64)
    yt = test_frame[['LONGITUDE','LATITUDE']].to_numpy(np.float64)
    with torch.no_grad():
        feature, base = baseline_features(result['model'], x)
        feature_test, base_test = baseline_features(result['model'], xt)
    assert feature.shape == (len(x), 64)
    yn = torch.from_numpy(scaler.transform(y).astype(np.float32))

    def dataset(indices):
        return TensorDataset(torch.from_numpy(x[indices]), feature[indices], base[indices], yn[indices])
    tr_loader = DataLoader(dataset(ti), batch_size=MIXER_BATCH_SIZE, shuffle=True)
    va_loader = DataLoader(dataset(vi), batch_size=MIXER_BATCH_SIZE, shuffle=False)
    te_loader = DataLoader(TensorDataset(torch.from_numpy(xt), feature_test, base_test,
                                        torch.from_numpy(scaler.transform(yt).astype(np.float32))),
                           batch_size=MIXER_BATCH_SIZE, shuffle=False)
    mixer = ResidualKANMixer().to(DEVICE)
    with torch.no_grad():
        rss0, feat0, base0, _ = next(iter(va_loader))
        assert torch.allclose(mixer(rss0.to(DEVICE), feat0.to(DEVICE)),
                              torch.zeros_like(base0).to(DEVICE), atol=1e-7)
    optimizer = torch.optim.AdamW(mixer.parameters(), lr=MIXER_LR, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',
                                                            factor=0.5, patience=5)
    loss_fn = LocalizationLoss(alpha=0.25)
    baseline_val, _ = mean_meters(base[vi].numpy(), y[vi], scaler)
    # alpha=0 là baseline và luôn là một lựa chọn hợp lệ theo validation.
    alphas = [0.0, 0.25, 0.5, 0.75, 1.0]
    best_val, best_state, best_alpha, best_epoch = baseline_val, None, 0.0, 0
    stale = 0
    for epoch in range(1, MIXER_EPOCHS+1):
        mixer.train(); total = 0.0
        for rss, feat, base_pred, target in tr_loader:
            rss, feat, base_pred, target = (v.to(DEVICE) for v in (rss,feat,base_pred,target))
            optimizer.zero_grad(set_to_none=True)
            pred = base_pred + mixer(rss, feat)
            loss = loss_fn(pred, target)
            if not torch.isfinite(loss): raise RuntimeError('Mixer loss is not finite')
            loss.backward()
            nn.utils.clip_grad_norm_(mixer.parameters(), 5.0)
            optimizer.step()
            total += loss.item()*len(rss)
        correction_val = mixer_predictions(mixer, va_loader, alpha=1.0) - base[vi].numpy()
        val_by_alpha = {a: mean_meters(base[vi].numpy()+a*correction_val, y[vi], scaler)[0]
                        for a in alphas}
        alpha = min(alphas, key=val_by_alpha.get)
        val_m = val_by_alpha[alpha]
        scheduler.step(val_m)
        if val_m < best_val - 1e-3:
            best_val, best_alpha, best_epoch = val_m, alpha, epoch
            best_state = copy.deepcopy(mixer.state_dict()); stale = 0
        else: stale += 1
        if epoch == 1 or epoch % 5 == 0:
            print(f'Tòa {bid} Mixer {epoch:03d}: train {total/len(ti):.4f} | '
                  f'val {val_m:.3f} m | baseline val {baseline_val:.3f} m | alpha {alpha}')
        if stale >= MIXER_PATIENCE: break

    if best_state is not None: mixer.load_state_dict(best_state)
    else: best_alpha = 0.0
    pred_norm = mixer_predictions(mixer, te_loader, best_alpha) if best_alpha else base_test.numpy()
    test_m, errors = mean_meters(pred_norm, yt, scaler)
    print(f'Tòa {bid}: baseline test {result["mean"]:.3f} m | '
          f'Mixer test {test_m:.3f} m | val {best_val:.3f} m | alpha {best_alpha}')
    torch.save({'baseline_state': result['model'].state_dict(),
                'mixer_state': mixer.state_dict() if best_state is not None else None,
                'scaler_mean': scaler.mean_, 'scaler_scale': scaler.scale_,
                'building': bid, 'alpha': best_alpha, 'best_epoch': best_epoch,
                'baseline_val_m': baseline_val, 'mixer_val_m': best_val,
                'baseline_test_m': result['mean'], 'mixer_test_m': test_m},
               SAVE_DIR/f'building_{bid}_bsrbf_residual_mixer.pt')
    return {'building': bid, 'baseline_test_m': result['mean'], 'mixer_test_m': test_m,
            'baseline_val_m': baseline_val, 'mixer_val_m': best_val,
            'alpha': best_alpha, 'epoch': best_epoch, 'errors': errors,
            'baseline_errors': result['errors']}


In [ ]:
# Official test chỉ in ở đây, sau khi chọn alpha và checkpoint bằng validation.
mixer_results = []
for result in baseline_results:
    mixer_results.append(train_residual_mixer(result))
    if torch.cuda.is_available(): torch.cuda.empty_cache()

comparison = pd.DataFrame([{k:v for k,v in r.items() if k not in ('errors','baseline_errors')}
                           for r in mixer_results])
display(comparison.round(4))
baseline_e = np.concatenate([r['baseline_errors'] for r in mixer_results])
mixer_e = np.concatenate([r['errors'] for r in mixer_results])
summary = {'metric': 'official UJI mean Euclidean distance (m)',
           'n_test': int(len(mixer_e)),
           'baseline_mean_m': float(baseline_e.mean()),
           'mixer_mean_m': float(mixer_e.mean()),
           'baseline_median_m': float(np.median(baseline_e)),
           'mixer_median_m': float(np.median(mixer_e)),
           'mixer_p90_m': float(np.percentile(mixer_e, 90)),
           'under_5m': bool(mixer_e.mean() < 5.0)}
print(json.dumps(summary, indent=2, ensure_ascii=False))
(SAVE_DIR/'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
comparison.to_csv(SAVE_DIR/'comparison_by_building.csv', index=False)

# Chẩn đoán các nhóm test: không dùng bảng này để chọn checkpoint/alpha.
diagnostics = []
for result in mixer_results:
    frame = test_df[test_df.BUILDINGID == result['building']].reset_index(drop=True)
    assert len(frame) == len(result['errors'])
    part = frame[['BUILDINGID','FLOOR','PHONEID']].copy()
    known_phones = set(train_df.loc[train_df.BUILDINGID == result['building'], 'PHONEID'])
    part['phone_seen_in_train'] = part['PHONEID'].isin(known_phones)
    part['baseline_error_m'] = result['baseline_errors']
    part['mixer_error_m'] = result['errors']
    diagnostics.append(part)
diagnostics = pd.concat(diagnostics, ignore_index=True)
by_phone = diagnostics.groupby(['BUILDINGID','PHONEID']).agg(
    n=('mixer_error_m','size'), baseline_m=('baseline_error_m','mean'),
    mixer_m=('mixer_error_m','mean')).reset_index()
by_floor = diagnostics.groupby(['BUILDINGID','FLOOR']).agg(
    n=('mixer_error_m','size'), baseline_m=('baseline_error_m','mean'),
    mixer_m=('mixer_error_m','mean')).reset_index()
by_seen_phone = diagnostics.groupby(['BUILDINGID','phone_seen_in_train']).agg(
    n=('mixer_error_m','size'), baseline_m=('baseline_error_m','mean'),
    mixer_m=('mixer_error_m','mean')).reset_index()
print('Điện thoại đã/chưa có trong train:')
display(by_seen_phone.round(3))
print('Sai số theo điện thoại (official test; chỉ để phân tích):')
display(by_phone.round(3))
print('Sai số theo tầng (official test; chỉ để phân tích):')
display(by_floor.round(3))
by_phone.to_csv(SAVE_DIR/'test_by_phone.csv', index=False)
by_floor.to_csv(SAVE_DIR/'test_by_floor.csv', index=False)
by_seen_phone.to_csv(SAVE_DIR/'test_by_seen_phone.csv', index=False)
